# GuppyLM — einen Transformer von Hand trainieren (Fassung B)

Wir bauen ein Sprachmodell mit rund zwölf Millionen Parametern (genaue Zahl nach Tokenizertraining), das sich wie ein kleiner
Fisch unterhält. Klein genug, für einen lokalen Trainingslauf, mit den grundlegenden Bauteilen eines Decoder-Transformers.

**Aufbau**

| Abschnitt | Inhalt |
|---|---|
| 1 | Umgebung einrichten |
| 2 | Datensatz laden, BPE-Tokenizer trainieren |
| 3 | Von Text zu Tensoren |
| 4 | Das Modell in vier Bausteinen |
| 5 | Training |
| 6 | Generierung |
| 7 | Gespräch mit dem Modell |
| 8 | Speichern und Laden |

**Architektur:** 6 Blöcke, `d_model` 384, 6 Köpfe, GELU-FFN mit Faktor 4, LayerNorm,
Positions-Embeddings (Abschnitt 4.4), BPE-Zielgröße 4096 (tatsächliche Größe wird gemessen). Kein RoPE, kein SwiGLU, kein GQA —
bewusst die schlichteste Variante, damit jedes Bauteil einzeln nachvollziehbar bleibt.

**Arbeitsauftrag:** Diese Fassung unterscheidet sich von
`guppylm_lehrnotebook.ipynb` in einem einzigen Baustein. Arbeiten Sie das
Notebook durch, finden Sie den Unterschied und begründen Sie, warum er so
umgesetzt ist. Leitfragen stehen am Ende unter *Weiterarbeiten*.

*Herkunft:* Bearbeitung von [arman-bd/guppylm](https://github.com/arman-bd/guppylm),
MIT-Lizenz laut README des Originals. Trainingsdaten:
[arman-bd/guppylm-60k-generic](https://huggingface.co/datasets/arman-bd/guppylm-60k-generic),
MIT, DOI 10.57967/hf/8339. Diese Fassung ist ins Deutsche übersetzt, didaktisch
neu aufgebaut und in der Architektur geändert (GELU statt ReLU, `ffn_hidden` 1536
statt 768). Einzelheiten in `notice.md`.

## 1. Umgebung

Starten Sie das Notebook mit seinem Ordner als Arbeitsverzeichnis. Prüfen Sie
unten den Pfad `WORK`. Dort liegen der lokale Datensatz und später die Checkpoints.
Architekturwerte werden zentral in `GRUNDKONFIG` gesetzt; die tatsächliche
Vokabulargröße und die Sondertoken-IDs werden nach dem Tokenizertraining übernommen.

PyTorch und der Trainingszyklus werden vorausgesetzt. Die Paketinstallation ist
nur bei der Einrichtung nötig und benötigt gegebenenfalls Internet; das Laden
des mitgelieferten Datensatzes selbst benötigt keinen Download.

Wir setzen sinnvolle positive Modellgrößen voraus. Die Teilbarkeit von
`d_model` durch `n_heads` prüfen wir ausdrücklich, weil sie für die
Aufteilung der Attention-Köpfe entscheidend ist.

In [ ]:
%pip install -q -r requirements-training.txt

In [ ]:
from pathlib import Path
import torch

# Arbeitsverzeichnis neben dem Notebook. Hier landen Daten und Checkpoints;
# importiert wird von hier nichts, alles Wesentliche steht im Notebook selbst.
# WORK = Path.cwd() / "guppy"
WORK = Path.cwd()
(WORK / "Checkpoints").mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

# Ausgangspunkt für den Kurs; Laufzeit und Qualität hängen von Hardware und Daten ab.
# Für die Kursplanung auf dem Zielgerät messen.
MAX_STEPS = 3000

print(f"PyTorch {torch.__version__}")
print(f"Arbeitsverzeichnis: {WORK}")
print(f"Device: {DEVICE}, Schritte: {MAX_STEPS}")

from dataclasses import dataclass, asdict, replace

@dataclass
class GuppyConfig:
    vocab_size: int = 4096    # Anzahl unterschiedlicher Tokens im Vokabular
    max_seq_len: int = 128    # Maximale Kontextlänge in Tokens
    d_model: int = 384        # Embeddingdimension und Breite des Residualstroms
    n_layers: int = 6         # Anzahl der Blöcke
    n_heads: int = 6          # Attention-Köpfe je Block
    ffn_hidden: int = 1536    # innere Breite des FFN (4 x d_model)
    dropout: float = 0.1      # Regularisierung im Training; bei model.eval() abgeschaltet
    pad_id: int = 0           # Token-ID von <pad> - das ist Fülltoken für batches.
    bos_id: int = 1           # <|im_start|> , beginning-of-sequence
    eos_id: int = 2           # <|im_end|> , end-of-sequence

    def __post_init__(self):
        # Diese Bedingung ist beim Experimentieren mit der Kopfzahl wichtig.
        if self.d_model % self.n_heads:
            raise ValueError("d_model muss durch n_heads teilbar sein.")

# Gemeinsame Architekturwerte: hier vor der Datenaufbereitung ändern.
GRUNDKONFIG = GuppyConfig()


## 2. Daten und Tokenizer

Der Datensatz enthält 60.000 einzelne Frage-Antwort-Paare zu 60 Themen:
Begrüßung, Futter, Wassertemperatur, Langeweile, Katzen vor dem Becken.
Fragen und Antworten sind **englisch**, die Antworten kurz und kleingeschrieben.

**Vorbereitung:** Im Ordner Data müssen `train.jsonl` (57.000 Beispiele) und `eval.jsonl`
(3.000 Beispiele) liegen. Beide enthalten bereits Texte im ChatML-Format.
Die Endung `.jsonl` bedeutet: ein JSON-Objekt pro Zeile.

Wir setzen diese vorbereiteten Dateien und das Arbeitsverzeichnis aus Abschnitt 1
voraus. Das Notebook liest sie direkt; ein Datendownload ist nicht erforderlich.

In [ ]:
import json


def lade_texte(dateipfad):
    texte = []
    with open(dateipfad, encoding="utf-8") as datei:
        for zeile in datei:
            beispiel = json.loads(zeile)
            texte.append(beispiel["text"])
    return texte


texte = {
    "train": lade_texte(WORK / "Data" / "train.jsonl"),
    "test": lade_texte(WORK / "Data" / "eval.jsonl"),
}

print(f"Training: {len(texte['train']):,} Beispiele")
print(f"Evaluation: {len(texte['test']):,} Beispiele")

### ChatML

Die lokal gespeicherten Beispiele enthalten bereits Rollenmarkierungen im
**ChatML-Format**. Damit lernt das Modell, **wo eine Antwort anfängt und aufhört**.
`<|im_start|>` und `<|im_end|>` werden beim Tokenizertraining als eigene
Sondertokens mit festen IDs angelegt.

Die entpackten Dateien enthalten pro Zeile ein JSON-Objekt mit dem Feld `text`.
Wir verwenden dessen Inhalt direkt; eine erneute Umwandlung ist nicht nötig.
Im Dictionary `texte` bezeichnet `"train"` die Trainingsdaten und `"test"`
die aus `eval.jsonl` geladenen Evaluationsdaten.

In [ ]:
print("Ein lokal geladenes Trainingsbeispiel im ChatML-Format:\n")
print(texte["train"][0])

### BPE-Tokenizer

Wir trainieren einen eigenen Byte-Level-BPE-Tokenizer ausschließlich auf den
Trainingsdaten. Die Zielgröße ist 4096; `min_frequency` und die verfügbaren
Merge-Paare können zu einem kleineren Vokabular führen. Das Modell verwendet
deshalb die anschließend gemessene Vokabulargröße.

Das vollständige Byte-Alphabet sorgt dafür, dass auch Großbuchstaben, Umlaute
und andere UTF-8-Zeichen kodiert werden können. Das bedeutet noch nicht,
dass das Sprachmodell ihre Bedeutung oder eine andere Sprache gelernt hat.

Die Sondertokens werden zuerst vergeben. Ihre IDs lesen wir vom Tokenizer ab.
Mit `replace` übernehmen wir die Grundkonfiguration und ändern nur die vier
Werte, die vom trainierten Tokenizer abhängen.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=GRUNDKONFIG.vocab_size,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    special_tokens=["<pad>", "<|im_start|>", "<|im_end|>"],
    min_frequency=2,
    show_progress=True,
)
tokenizer.train_from_iterator(texte["train"], trainer)
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
tokenizer.save(str(WORK / "data" / "tokenizer.json"))

print(f"Vokabular: {tokenizer.get_vocab_size()} Tokens")

# Vokabular und Sondertoken-IDs stammen aus genau diesem Tokenizer.
konfig = replace(
    GRUNDKONFIG,
    vocab_size=tokenizer.get_vocab_size(),
    pad_id=tokenizer.token_to_id("<pad>"),
    bos_id=tokenizer.token_to_id("<|im_start|>"),
    eos_id=tokenizer.token_to_id("<|im_end|>"),
)

for text in ("hello guppy", "Quantenverschränkung", "Hello, café! 🐟"):
    assert tokenizer.decode(tokenizer.encode(text).ids) == text, text
print("Rückübersetzung einschließlich Großbuchstaben und UTF-8 geprüft.")


### Beobachtung: Was macht der Tokenizer?

Unten steht links die Token-ID, rechts das Stück Text, für das sie steht.
Achten Sie auf drei Dinge:

- `Ġ` markiert ein führendes Leerzeichen. `Ġdu` und `du` sind verschiedene Tokens.
- `Ċ` ist ein Zeilenumbruch.
- Die Rollenmarkierungen sind **ein** Token, nicht zwölf Zeichen.

Ändern Sie `probe` in etwas, das nicht im Datensatz vorkommt — etwa
`"Quantenverschränkung"`. Prüfen Sie die Rückübersetzung. Unterscheiden Sie die
Kodierbarkeit des Textes von der Fähigkeit des Modells, darauf sinnvoll zu antworten.

In [ ]:
probe = "guppy likes bubbles"
kodiert = tokenizer.encode(probe)

print(f"{probe!r}  ->  {len(kodiert.ids)} Tokens\n")
for tid, stueck in zip(kodiert.ids, kodiert.tokens):
    print(f"  {tid:5d}   {stueck!r}")

print("\nSondertokens:")
for name in ["<pad>", "<|im_start|>", "<|im_end|>"]:
    print(f"  {name:14s} -> ID {tokenizer.token_to_id(name)}")

print(f"\nZurueckuebersetzt: {tokenizer.decode(kodiert.ids)!r}")

### Beobachtung: Wie lang ist ein Beispiel?

Die maximale Sequenzlänge `max_seq_len` ist keine willkürliche Zahl. Sie muss
groß genug sein, dass Beispiele nicht abgeschnitten werden, und klein genug,
dass Rechenzeit und Positions-Embeddings nicht verschwendet werden. Der
Rechenaufwand der Attention wächst **quadratisch** mit dieser Länge.

Lesen Sie das 99-Prozent-Quantil ab und prüfen Sie die gewählte Kontextlänge.
Ein Beispiel mit L Tokens ergibt L-1 Eingabepositionen. Die tatsächliche
Batchlänge ergibt sich durch dynamisches Padding, nicht durch Auffüllen auf die Kontextgrenze.

In [ ]:
import matplotlib.pyplot as plt

laengen = sorted(len(tokenizer.encode(t).ids) for t in texte["train"])
for q in (0.5, 0.95, 0.99, 1.0):
    position = min(int(q * len(laengen)), len(laengen) - 1)
    print(f"  {q:5.0%}-Quantil: {laengen[position]:3d} Tokens")

plt.figure(figsize=(7, 3))
plt.hist(laengen, bins=40)
plt.axvline(konfig.max_seq_len + 1, color="crimson",
            label=f"max. Beispiel = Kontext + 1 = {konfig.max_seq_len + 1}")
plt.xlabel("Tokens je Beispiel")
plt.ylabel("Anzahl")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Von Text zu Tensoren

Ein Sprachmodell sagt zu jeder Position das **nächste** Token voraus. Aus einer
Sequenz werden deshalb zwei: die Eingabe `x` ohne das letzte Token, das Ziel `y`
ohne das erste. Beide sind gleich lang und um genau eine Position gegeneinander
verschoben.

Weil Beispiele unterschiedlich lang sind, füllen wir kürzere mit `<pad>` (ID 0)
auf. Diese Füllung darf den Verlust nicht beeinflussen — dafür sorgt später
`ignore_index=0`.

In [ ]:
def zu_sequenzen(liste, max_len):
    # Ein Beispiel benötigt L-1 Eingabepositionen. Deshalb sind L <= Kontext+1 zulässig.
    ergebnis = []
    for nummer, text in enumerate(liste):
        ids = tokenizer.encode(text).ids
        if len(ids) > max_len + 1:
            raise ValueError(
                f"Beispiel {nummer} hat {len(ids)} Tokens; für Kontext {max_len} "
                "sind höchstens max_seq_len + 1 zulässig. Kontext vergrößern "
                "oder die Daten bewusst aufbereiten; es wird nichts still abgeschnitten."
            )
        if len(ids) >= 2:
            ergebnis.append(ids)
    return ergebnis

train_seq = zu_sequenzen(texte["train"], konfig.max_seq_len)
eval_seq = zu_sequenzen(texte["test"], konfig.max_seq_len)
print(f"{len(train_seq):,} Trainings-, {len(eval_seq):,} Evaluationssequenzen")


def baue_batch(quelle, indizes, pad_id=None):
    """Baut aus Sequenz-Indizes einen rechts gepaddeten Batch (x, y)."""
    if pad_id is None:
        pad_id = konfig.pad_id
    sequenzen = [quelle[i] for i in indizes]
    batch_size = len(sequenzen)
    laenge = max(len(ids) - 1 for ids in sequenzen)
    x = torch.full((batch_size, laenge), pad_id, dtype=torch.long)
    y = torch.full((batch_size, laenge), pad_id, dtype=torch.long)

    for zeile, ids in enumerate(sequenzen):
        eingabe = torch.tensor(ids[:-1])
        ziel = torch.tensor(ids[1:])
        x[zeile, :len(eingabe)] = eingabe
        y[zeile, :len(ziel)] = ziel
    return x, y


### Beobachtung: Der Versatz um eine Position

Vergleichen Sie die beiden Zeilen Spalte für Spalte. Das Ziel an Position i
ist das nächste Token der ursprünglichen Sequenz.

Hier wird ausschließlich **rechts** gepaddet. Durch die kausale Maske können
gültige Positionen das nachfolgende Padding nicht sehen; die Ziele an
Paddingpositionen ignoriert der Verlust. Eine zusätzliche Padding-Maske ist
für genau diesen Aufbau nicht nötig. Bei linkem Padding oder nichtkausaler
Attention muss diese Annahme neu geprüft werden.

In [ ]:
x, y = baue_batch(train_seq, [0, 1, 2, 3])
print("Form:", tuple(x.shape), "\n")
print("x[0]:", x[0][:12].tolist())
print("y[0]:", y[0][:12].tolist())
print("\nals Text:")
print("  x:", repr(tokenizer.decode(x[0][:12].tolist(), skip_special_tokens=False)))
print("  y:", repr(tokenizer.decode(y[0][:12].tolist(), skip_special_tokens=False)))
print(f"\nPadding-Anteil in diesem Batch: {(y == konfig.pad_id).float().mean():.1%}")

## 4. Das Modell

Wir bauen den Transformer in vier Schritten: **Attention**, **FFN**, **Block**,
**Gesamtmodell**. Nach jedem Schritt steht eine Beobachtungszelle, die das gerade
gebaute Teil einzeln ausprobiert. Erst am Ende setzen wir alles zusammen.

In [ ]:
# Die Konfiguration steht bereits vor der Datenaufbereitung fest.
print(konfig)
print(f"\nDimension je Kopf: {konfig.d_model} / {konfig.n_heads} = "
      f"{konfig.d_model // konfig.n_heads}")

### 4.1 Attention

Jedes Token bildet drei Vektoren: **Query** (wonach suche ich?), **Key**
(was biete ich an?) und **Value** (was gebe ich weiter?). Das Skalarprodukt aus
Query und Key ergibt für jedes Tokenpaar eine Ähnlichkeit; nach Skalierung mit
$\sqrt{d_{\text{head}}}$ und Softmax werden daraus Gewichte, mit denen die Values
gemittelt werden.

Entscheidend ist die **kausale Maske**: Eine Position darf sich selbst und frühere Positionen sehen.
Ohne sie könnte das Modell die Antwort einfach ablesen, statt sie vorherzusagen.

Das Argument `return_attn` gibt zusätzlich die Gewichtsmatrix zurück. Das Modell
braucht es nicht — es existiert nur, damit wir gleich hineinsehen können.

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F


def kausale_maske(T, device=None):
    """Untere Dreiecksmatrix: Position i sieht nur 0..i."""
    return torch.tril(torch.ones(T, T, device=device)).view(1, 1, T, T)


class Attention(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.n_heads = konfig.n_heads
        self.head_dim = konfig.d_model // konfig.n_heads
        self.qkv = nn.Linear(konfig.d_model, 3 * konfig.d_model)
        self.out = nn.Linear(konfig.d_model, konfig.d_model)
        self.dropout = nn.Dropout(konfig.dropout)

    def forward(self, x, maske=None, return_attn=False):
        B, T, C = x.shape

        # eine Matrixmultiplikation fuer Q, K und V zusammen, dann aufteilen
        qkv = self.qkv(x)
        qkv = qkv.reshape(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)      # je (B, Koepfe, T, head_dim)

        roh = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if maske is not None:
            roh = roh.masked_fill(maske == 0, float("-inf"))
        gewichte = F.softmax(roh, dim=-1)          # (B, Koepfe, T, T)

        y = self.dropout(gewichte) @ v    # Values je Kopf gewichten
        y = y.transpose(1, 2)               # Tokenposition vor Kopfnummer
        y = y.contiguous().view(B, T, C)    # Köpfe wieder zusammenführen
        y = self.out(y)
        if return_attn:
            return y, gewichte
        return y

### Beobachtung: Was die Maske bewirkt

Unten sehen Sie erst die Maske selbst, dann eine Zeile der Attention-Gewichte
einmal ohne und einmal mit Maske. In der maskierten Fassung stehen rechts von der
aktuellen Position Nullen — dorthin darf nicht geschaut werden. Beide Zeilen
summieren sich trotzdem zu 1, weil Softmax nach dem Maskieren normiert.

Zum Ausprobieren: Setzen Sie `position` auf 0 und auf `T-1`. Wie viele Tokens
stehen dem ersten Token zur Verfügung, wie viele dem letzten?

In [ ]:
torch.manual_seed(0)
T, position = 6, 2

probe_attn = Attention(konfig).eval()
probe_x = torch.randn(1, T, konfig.d_model)
maske = kausale_maske(T)

print("Kausale Maske:")
print(maske[0, 0].to(torch.int).numpy())

_, ohne = probe_attn(probe_x, None,  return_attn=True)
_, mit  = probe_attn(probe_x, maske, return_attn=True)

print(f"\nEingabe {tuple(probe_x.shape)}  ->  Gewichte {tuple(mit.shape)}")
print("  (Batch, Koepfe, Ziel-Position, Quell-Position)\n")
print(f"Zeile {position} ohne Maske: {[round(w, 3) for w in ohne[0, 0, position].tolist()]}")
print(f"Zeile {position} mit  Maske: {[round(w, 3) for w in mit[0, 0, position].tolist()]}")
print(f"\nZeilensumme mit Maske: {mit[0, 0, position].sum():.4f}")

### 4.2 Das Feed-Forward-Netz

Die Attention mischt Information **zwischen** Positionen. Das FFN verarbeitet
jede Position **für sich**: hochprojizieren, Aktivierung, zurückprojizieren. Hier
sitzt der größere Teil der Parameter eines Blocks.

Als Aktivierung nehmen wir **GELU** in der tanh-Näherung — dieselbe Funktion, die
GPT-2 unter dem Namen `gelu_new` verwendet. ReLU täte es didaktisch auch, aber die
Wahl ist hier nicht frei: Wer das Modell später in LM Studio laden will, muss GELU
nehmen, weil llama.cpp die Aktivierung für diese Architektur fest eingebaut hat.

**Zur Diskussion:** `ffn_hidden` ist das Vierfache von `d_model` — der übliche
Faktor. Rechnen Sie unten nach, wie viele Parameter der Faktor 2 sparen würde,
und überlegen Sie, was ein kleineres FFN bei einem so engen Wortschatz verliert.
Auch hier gilt die Einschränkung von oben: Der Weg nach LM Studio verlangt exakt
Faktor 4.

In [ ]:
class FFN(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.hoch   = nn.Linear(konfig.d_model, konfig.ffn_hidden)
        self.runter = nn.Linear(konfig.ffn_hidden, konfig.d_model)
        self.dropout = nn.Dropout(konfig.dropout)

    def forward(self, x):
        # tanh-Naeherung: exakt das, was GPT-2 als "gelu_new" rechnet
        x = self.hoch(x)
        x = F.gelu(x, approximate="tanh")
        x = self.runter(x)
        return self.dropout(x)


def zaehle(modul):
    return sum(p.numel() for p in modul.parameters())


print(f"FFN mit Faktor {konfig.ffn_hidden / konfig.d_model:.0f}: "
      f"{zaehle(FFN(konfig)):,} Parameter")
kleine_konfig = replace(konfig, ffn_hidden=2 * konfig.d_model)
kleines_ffn = FFN(kleine_konfig)
print(f"FFN mit Faktor 2: {zaehle(kleines_ffn):,} Parameter")

### 4.3 Der Block

Attention und FFN addieren ihre Beiträge zum Residualstrom. Die Residualpfade
erleichtern den Gradientenfluss; sie garantieren keine beliebig stabile Optimierung.

Wir normieren vor jedem Baustein (Pre-Norm). Das unterstützt stabiles Training.
Zusätzlich verwenden wir im Trainingsabschnitt einen Lernraten-Warmup.

In [ ]:
class Block(nn.Module):
    def __init__(self, konfig):
        super().__init__()
        self.norm1 = nn.LayerNorm(konfig.d_model)
        self.attn  = Attention(konfig)
        self.norm2 = nn.LayerNorm(konfig.d_model)
        self.ffn   = FFN(konfig)

    def forward(self, x, maske=None):
        x = x + self.attn(self.norm1(x), maske)   # Beitrag, kein Ersatz
        x = x + self.ffn(self.norm2(x))
        return x


torch.manual_seed(0)
probe_block = Block(konfig).eval()
with torch.no_grad():
    nachher = probe_block(probe_x, maske)

aenderung = (nachher - probe_x).norm() / probe_x.norm()
print(f"Norm des Residualstroms vorher : {probe_x.norm():.3f}")
print(f"Norm des Residualstroms nachher: {nachher.norm():.3f}")
print(f"relative Aenderung durch den Block: {aenderung:.1%}")
print(f"\nParameter je Block: {zaehle(probe_block):,}")

### 4.4 Das Gesamtmodell

Token-Embedding plus Positions-Embedding ergeben den Anfangszustand, dann sechs
Blöcke, eine abschließende Normierung und die Projektion zurück auf das Vokabular.

Eine Besonderheit: `lm_head` und `tok_emb` **teilen sich dieselbe Gewichtsmatrix**
(weight tying). Die Zeile `self.lm_head.weight = self.tok_emb.weight` ist keine
Kopie, sondern dieselbe Matrix unter zwei Namen. Es handelt sich um eine gemeinsame Parametrisierung von Eingabe und Ausgabe;
die Ausgabeprojektion ist keine mathematische Umkehrfunktion des Embeddings.

#### Positionskodierung und Skalierungsfaktor α

Die Positionswerte werden in dieser Fassung **nicht gelernt**, sondern nach
Vaswani et al. (2017) aus Sinus- und Kosinusfunktionen verschiedener Frequenzen
berechnet und eingefroren. Die Tabelle wird zusätzlich mit einem festen Faktor
**α = 0,02 · √2 ≈ 0,0283** multipliziert.

*Begründung für α:*

- Ohne Skalierung haben die Einträge der Tabelle einen quadratischen Mittelwert
  (RMS) von 1/√2 ≈ 0,71. Ein Positionsvektor hat damit die Länge √192 ≈ 13,9,
  ein frisch initialisiertes Token-Embedding (Standardabweichung 0,02) nur
  0,02 · √384 ≈ 0,39. Die Position wäre am Eingang rund 35-mal stärker als das Token.
- Das Original gleicht das aus, indem es die Token-Embeddings mit √`d_model`
  multipliziert. Das scheidet hier aus: Wegen des weight tying würde derselbe
  Faktor die Ausgabelogits verändern, und die GPT-2-Architektur in llama.cpp,
  die der Export nach LM Studio nutzt, kennt diesen Faktor nicht.
- Deshalb wird stattdessen die Positionstabelle verkleinert. α = 0,02 · √2 gibt
  jedem Eintrag den RMS 0,02, also exakt die Größenordnung, mit der die
  Token-Embeddings starten. Token- und Positionssignal beginnen gleich stark.

*Einschränkung:* α ist eine begründete Setzung, kein in der Literatur etablierter
Standardwert und nicht durch Hyperparametersuche bestimmt. Die Token-Embeddings
können im Training wachsen, die eingefrorene Tabelle nicht. Ob ein anderes α
besser trainiert, ist eine empirische Frage (siehe *Weiterarbeiten*).

In [ ]:
# Skalierungsfaktor alpha der Sinustabelle (Begründung im Markdown von 4.4).
# Unskaliert besteht jede Zeile aus d_model/2 Paaren (sin, cos) mit sin² + cos² = 1;
# der quadratische Mittelwert je Eintrag ist daher genau 1/2, der RMS 1/√2 ≈ 0,707.
# Mit alpha = 0,02·√2 hat jeder Eintrag den RMS 0,02 – dieselbe Größenordnung wie
# die mit Standardabweichung 0,02 initialisierten Token-Embeddings.
POS_ALPHA = 0.02 * math.sqrt(2)


def sinus_tabelle(max_len, d_model, alpha=POS_ALPHA):
    """Sinusförmige Positionskodierung nach Vaswani et al. (2017), mit alpha skaliert.

    Gerade Dimensionen erhalten den Sinus, ungerade den Kosinus derselben Frequenz.
    """
    position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)    # (max_len, 1)
    i = torch.arange(0, d_model, 2, dtype=torch.float32)                  # 0, 2, 4, ...
    winkel = position / (10000 ** (i / d_model))                          # (max_len, d_model/2)
    tabelle = torch.zeros(max_len, d_model)
    tabelle[:, 0::2] = torch.sin(winkel)
    tabelle[:, 1::2] = torch.cos(winkel)
    return alpha * tabelle


class GuppyLM(nn.Module):
    def __init__(self, konfig: GuppyConfig):
        super().__init__()
        self.konfig = konfig
        self.tok_emb = nn.Embedding(konfig.vocab_size, konfig.d_model)
        self.pos_emb = nn.Embedding(konfig.max_seq_len, konfig.d_model)
        self.drop = nn.Dropout(konfig.dropout)
        self.blocks = nn.ModuleList([Block(konfig) for _ in range(konfig.n_layers)])
        self.norm = nn.LayerNorm(konfig.d_model)
        self.lm_head = nn.Linear(konfig.d_model, konfig.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight     # dieselbe Matrix, nicht eine Kopie
        self.apply(self._init_gewichte)

        # Positionen: feste Tabelle statt gelernter Werte. pos_emb bleibt ein
        # nn.Embedding, damit Checkpointformat und Export unverändert bleiben;
        # die Tabelle wird nach der Initialisierung eingesetzt und eingefroren.
        with torch.no_grad():
            self.pos_emb.weight.copy_(sinus_tabelle(konfig.max_seq_len, konfig.d_model))
        self.pos_emb.weight.requires_grad_(False)

    def _init_gewichte(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, ziele=None):
        B, T = idx.shape
        if not 1 <= T <= self.konfig.max_seq_len:
            raise ValueError(f"Sequenzlänge {T} außerhalb des Kontexts {self.konfig.max_seq_len}.")
        pos = torch.arange(T, device=idx.device)
        token_embeddings = self.tok_emb(idx)
        positions_embeddings = self.pos_emb(pos)
        x = self.drop(token_embeddings + positions_embeddings)

        maske = kausale_maske(T, idx.device)
        for block in self.blocks:
            x = block(x, maske)

        logits = self.lm_head(self.norm(x))

        verlust = None
        if ziele is not None:
            verlust = F.cross_entropy(
                logits.view(-1, self.konfig.vocab_size),
                ziele.reshape(-1),
                ignore_index=self.konfig.pad_id,   # Padding zaehlt nicht
            )
        return logits, verlust

### Beobachtung: Wo sitzen die Parameter?

Bevor Sie die Zelle ausführen: Schätzen Sie, welcher Anteil auf die Embeddings
und welcher auf die sechs Blöcke entfällt. Die meisten schätzen die Embeddings
zu hoch.

In [ ]:
modell = GuppyLM(konfig)
gesamt = zaehle(modell)

print(f"{'Bauteil':<12}{'Parameter':>12}{'Anteil':>9}")
print("-" * 33)
for name, modul in [("tok_emb", modell.tok_emb), ("pos_emb", modell.pos_emb),
                    ("blocks", modell.blocks), ("norm", modell.norm)]:
    print(f"{name:<12}{zaehle(modul):>12,}{zaehle(modul)/gesamt:>9.1%}")
print("-" * 33)
print(f"{'gesamt':<12}{gesamt:>12,}{'':>9}")
trainierbar = sum(p.numel() for p in modell.parameters() if p.requires_grad)
print(f"{'trainierbar':<12}{trainierbar:>12,}{trainierbar/gesamt:>9.1%}")

print(f"\nOhne weight tying waeren es {gesamt + zaehle(modell.tok_emb):,} Parameter —")
print(f"die gemeinsame Matrix spart {zaehle(modell.tok_emb)/(gesamt+zaehle(modell.tok_emb)):.0%}.")

### Beobachtung: Wie sehen die Positions-Embeddings aus?

Die Zelle zeigt die Positionswerte als Bild: jede Zeile eine Position, jede
Spalte eine Dimension. Führen Sie dieselbe Zelle auch in
`guppylm_lehrnotebook.ipynb` aus, einmal vor und einmal nach dem Training.
Beschreiben Sie, was sich unterscheidet und was sich durch das Training ändert.

In [ ]:
werte = modell.pos_emb.weight.detach().cpu()
grenze = werte.abs().max().item()

plt.figure(figsize=(7, 3))
plt.imshow(werte.numpy(), aspect="auto", cmap="RdBu_r", vmin=-grenze, vmax=grenze)
plt.colorbar(label="Wert")
plt.xlabel("Dimension")
plt.ylabel("Position")
plt.title("Positions-Embeddings")
plt.tight_layout()
plt.show()
print(f"RMS je Eintrag: {werte.pow(2).mean().sqrt():.4f}, "
      f"trainierbar: {modell.pos_emb.weight.requires_grad}")


### 4.5 Sanity-Check: Eine Referenz für den Startverlust

Eine Gleichverteilung über V Tokens hat Kreuzentropie `ln(V)`. Verwenden Sie
hier die tatsächliche Vokabulargröße. Ein zufällig initialisiertes Netz liefert
keine exakt gleichförmige Verteilung; sein Verlust muss daher nicht genau
`ln(V)` entsprechen. Auffällige Abweichungen sind Anlass zur Prüfung, aber
allein kein Beweis für Informationsleckage oder defekte Initialisierung.

In [ ]:
modell.eval()
x_probe, y_probe = baue_batch(train_seq, list(range(8)))
with torch.no_grad():
    _, start_verlust = modell(x_probe, y_probe)

print(f"Verlust bei Zufallsgewichten: {start_verlust:.3f}")
print(f"ln(vocab_size)              : {math.log(konfig.vocab_size):.3f}")
print(f"Abweichung                  : {abs(start_verlust - math.log(konfig.vocab_size)):.3f}")

## 5. Training

Die Schleife steht offen im Notebook, nicht in einer Funktion. Pro Schritt:
zufälligen Batch ziehen, Verlust berechnen, Gradienten zurückrechnen, Gewichte
anpassen.

Drei Details, die nicht selbstverständlich sind:

- **Lernraten-Zeitplan.** Erst linear hoch (Warmup), dann als Kosinuskurve
  herunter. Der Zeitplan wird unten geplottet, bevor er benutzt wird.
- **Gradient Clipping.** Begrenzt die Länge des Gradientenvektors auf 1,0 und
  kann extreme Gradientenausschläge begrenzen. Es begrenzt nicht unmittelbar
  die von AdamW ausgeführte Gewichtsänderung.
- **Feste Evaluationsbatches.** Sie werden einmal gezogen und bleiben gleich —
  das reduziert die Schwankung durch wechselnde Stichproben.

In [ ]:
BATCH_SIZE   = 32
LR           = 3e-4
MIN_LR       = 3e-5
WARMUP       = 200
EVAL_ALLE    = 200
EVAL_BEISPIELE = 320
WEIGHT_DECAY = 0.1
GRAD_CLIP    = 1.0


def lernrate(schritt):
    if schritt < WARMUP:
        return LR * (schritt + 1) / WARMUP
    fortschritt = (schritt - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return MIN_LR + (LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * fortschritt))


plt.figure(figsize=(7, 2.5))
plt.plot([lernrate(s) for s in range(MAX_STEPS)])
plt.axvline(WARMUP, color="crimson", ls="--", label=f"Ende Warmup ({WARMUP})")
plt.xlabel("Schritt")
plt.ylabel("Lernrate")
plt.legend()
plt.tight_layout()
plt.show()

### Feste Batches für die Validierung

Wir bereiten die ersten `EVAL_BEISPIELE` Evaluationssequenzen einmal vor.
So wird während des Trainings immer dieselbe Stichprobe bewertet.

In [ ]:
import time

torch.manual_seed(42)

# Feste Validierungsstichprobe, begrenzt für einen kurzen Lehrlauf.
eval_batches = []
anzahl_eval = min(EVAL_BEISPIELE, len(eval_seq))
for start in range(0, anzahl_eval, BATCH_SIZE):
    indizes = list(range(start, min(start + BATCH_SIZE, anzahl_eval)))
    bx, by = baue_batch(eval_seq, indizes)
    eval_batches.append((bx.to(DEVICE), by.to(DEVICE)))
print(f"Validierung auf {anzahl_eval} Beispielen.")

### Den Validierungsverlust berechnen

Die Funktion schaltet Dropout aus und mittelt den Verlust über alle gültigen
Zieltoken. Batches mit mehr Text erhalten dadurch entsprechend mehr Gewicht.
Anschließend stellt sie den vorherigen Trainingsmodus wieder her.

In [ ]:
@torch.no_grad()
def bewerte(modell):
    war_training = modell.training
    modell.eval()
    verlustsumme = 0.0
    tokenanzahl = 0
    for x, y in eval_batches:
        _, verlust = modell(x, y)
        gueltige_tokens = (y != modell.konfig.pad_id).sum().item()
        verlustsumme += verlust.item() * gueltige_tokens
        tokenanzahl += gueltige_tokens

    modell.train(war_training)
    return verlustsumme / tokenanzahl

### Das Modell trainieren

Jetzt folgen Initialisierung und Trainingsschleife. Bei einer Verbesserung des
Validierungsverlusts speichern wir Gewichte, Konfiguration und Tokenizer gemeinsam.

In [ ]:
# Tokenizer als Bestandteil jedes Checkpoints, nicht nur als separate Datei.
import hashlib
tokenizer_json = tokenizer.to_str()
tokenizer_sha256 = hashlib.sha256(tokenizer_json.encode("utf-8")).hexdigest()

modell = GuppyLM(konfig).to(DEVICE)
optimierer = torch.optim.AdamW(modell.parameters(), lr=LR,
                               weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
modell.train()

verlaufsverlust = []
eval_punkte = []
bester_eval = float("inf")
t0 = time.time()

print(f"{'Schritt':>8} | {'Lernrate':>9} | {'Train':>8} | {'Eval':>8} | {'Zeit':>7}")
print("-" * 52)

for schritt in range(MAX_STEPS):
    indizes = torch.randint(0, len(train_seq), (BATCH_SIZE,)).tolist()
    x, y = baue_batch(train_seq, indizes)
    x, y = x.to(DEVICE), y.to(DEVICE)

    lr = lernrate(schritt)
    for gruppe in optimierer.param_groups:
        gruppe["lr"] = lr

    _, verlust = modell(x, y)
    verlust.backward()
    torch.nn.utils.clip_grad_norm_(modell.parameters(), GRAD_CLIP)
    optimierer.step()
    optimierer.zero_grad(set_to_none=True)
    verlaufsverlust.append(verlust.item())

    if schritt % EVAL_ALLE == 0 or schritt == MAX_STEPS - 1:
        ev = bewerte(modell)
        eval_punkte.append((schritt, ev))
        letzte_verluste = verlaufsverlust[-EVAL_ALLE:]
        mittel = sum(letzte_verluste) / len(letzte_verluste)
        print(f"{schritt:8d} | {lr:9.6f} | {mittel:8.4f} | {ev:8.4f} | "
              f"{time.time()-t0:6.1f}s")
        if ev < bester_eval:
            bester_eval = ev
            checkpoint = {
                "schritt": schritt,
                "model_state_dict": modell.state_dict(),
                "konfig": asdict(konfig),
                "eval_verlust": ev,
                "tokenizer_json": tokenizer_json,
                "tokenizer_sha256": tokenizer_sha256,
            }
            torch.save(checkpoint, WORK / "checkpoints" / "bestes_modell_b.pt")

print(f"\nFertig in {time.time()-t0:.0f}s. Bester Eval-Verlust: {bester_eval:.4f}")

### Beobachtung: Die Verlustkurve lesen

Der Trainingsverlust stammt aus wechselnden Batches mit aktivem Dropout;
der Validierungsverlust wird ohne Dropout am aktuellen Modellstand gemessen.
Der gleitende Mittelwert enthält zudem frühere Modellstände. Deshalb kann
die Validierung unter der Trainingskurve liegen.

Steigender Validierungsverlust bei sinkendem Trainingsverlust kann auf
Überanpassung hindeuten. Gemeinsam fallende Kurven belegen allein kein
Sprachverständnis und keine Generalisierung auf neue Fragen.

**Datengrenze:** Im lokalen Archiv sind 2.320 von 3.000 Evaluationsdialogen
bereits wortgleich im Training enthalten; unter den ersten 320 sind es 248.
Dieser Split wird zur Checkpointauswahl verwendet und ist damit eine
Validierung, kein unabhängiger Abschlusstest. Ergänzen Sie die Beurteilung
später durch neue englische Formulierungen.

Die Linie `ln(V)` ist die Referenz einer Gleichverteilung, kein garantierter Startwert.

In [ ]:
fenster = 50
geglaettet = []
for i in range(len(verlaufsverlust)):
    start = max(0, i - fenster + 1)
    ausschnitt = verlaufsverlust[start:i + 1]
    geglaettet.append(sum(ausschnitt) / len(ausschnitt))

plt.figure(figsize=(8, 4))
plt.plot(verlaufsverlust, alpha=0.2, label="Verlust je Batch")
plt.plot(geglaettet, label=f"gleitender Mittelwert ({fenster})")
plt.plot(*zip(*eval_punkte), "o-", label="Evaluation")
plt.axhline(math.log(konfig.vocab_size), color="gray", ls="--",
            label=f"Gleichverteilung: ln({konfig.vocab_size})")
plt.xlabel("Schritt")
plt.ylabel("Kreuzentropie")
plt.legend()
plt.tight_layout()
plt.show()

### Das beste Modell für die Antworttests laden

Wir verwenden für alle folgenden Antworten den besten gespeicherten Stand und seinen Tokenizer.

In [ ]:
import hashlib
from tokenizers import Tokenizer


def lade_checkpoint(pfad, device):
    ckpt = torch.load(pfad, map_location="cpu", weights_only=True)
    if "tokenizer_json" not in ckpt or "tokenizer_sha256" not in ckpt:
        raise ValueError("Alter Checkpoint ohne zugeordneten Tokenizer. Bitte neu trainieren.")
    inhalt = ckpt["tokenizer_json"]
    pruefsumme = hashlib.sha256(inhalt.encode("utf-8")).hexdigest()
    if pruefsumme != ckpt["tokenizer_sha256"]:
        raise ValueError("Tokenizer-Prüfsumme im Checkpoint stimmt nicht.")
    tok = Tokenizer.from_str(inhalt)
    cfg = GuppyConfig(**ckpt["konfig"])
    if tok.get_vocab_size() != cfg.vocab_size:
        raise ValueError("Tokenizer und Modell haben verschiedene Vokabulargrößen.")
    sondertokens = {
        "<pad>": cfg.pad_id,
        "<|im_start|>": cfg.bos_id,
        "<|im_end|>": cfg.eos_id,
    }
    for name, token_id in sondertokens.items():
        if tok.token_to_id(name) != token_id:
            raise ValueError(f"Sondertoken {name} passt nicht zur Modellkonfiguration.")
    netz = GuppyLM(cfg).to(device)
    netz.load_state_dict(ckpt["model_state_dict"])
    netz.eval()
    return netz, tok, ckpt


modell, tokenizer, ckpt = lade_checkpoint(WORK / "checkpoints" / "bestes_modell_b.pt", DEVICE)
print(f"Für die Antworttests: bester Checkpoint aus Schritt {ckpt['schritt']}, "
      f"{sum(p.numel() for p in modell.parameters()):,} Parameter.")


## 6. Generierung

Trainiert wird auf ganzen Sequenzen parallel, erzeugt wird Token für Token: Das
Modell liefert Logits, daraus wird ein Token gezogen, das Token wird angehängt,
und das Ganze beginnt von vorn. Genau diese Schleife macht Textgenerierung
langsam — jedes Token kostet einen vollständigen Durchlauf.

Zwei Stellschrauben steuern, wie mutig gezogen wird:

- **Temperatur** teilt die Logits. Kleine Werte spitzen die Verteilung zu
  (vorhersagbar, wiederholend), große Werte flachen sie ab (kreativ, wirr).
- **top-k** streicht alle außer den `k` wahrscheinlichsten Tokens. Das verhindert,
  dass ein sehr unwahrscheinliches Token die Antwort entgleisen lässt.

In [ ]:
import warnings


@torch.no_grad()
def erzeuge(modell, ids, max_neu=64, temperatur=0.7, top_k=50, return_info=False):
    """Generiert innerhalb des Kontextfensters; optional mit Abbruchgrund."""
    if not math.isfinite(temperatur) or temperatur < 0:
        raise ValueError("Die Temperatur muss endlich und mindestens 0 sein.")
    if not isinstance(max_neu, int) or max_neu < 0:
        raise ValueError("max_neu muss eine nichtnegative ganze Zahl sein.")
    if top_k is not None and (not isinstance(top_k, int) or top_k < 0):
        raise ValueError("top_k muss None oder eine nichtnegative ganze Zahl sein.")
    if not ids or len(ids) > modell.konfig.max_seq_len:
        raise ValueError(f"Der Prompt benötigt 1 bis {modell.konfig.max_seq_len} Tokens.")
    modell.eval()
    device = next(modell.parameters()).device
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    grund = "Tokenlimit"

    for _ in range(max_neu):
        if idx.size(1) >= modell.konfig.max_seq_len:
            grund = "Kontextgrenze"
            break
        logits = modell(idx)[0][:, -1, :].clone()
        # Eine einzelne Assistant-Antwort enthält weder Padding noch einen Rollenstart.
        logits[:, [modell.konfig.pad_id, modell.konfig.bos_id]] = float("-inf")
        if temperatur == 0:
            naechstes = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperatur
            if top_k:
                k = min(top_k, logits.size(-1))
                top_werte = torch.topk(logits, k).values
                grenze = top_werte[:, -1:]
                logits = logits.masked_fill(logits < grenze, float("-inf"))
            wahrscheinlichkeiten = F.softmax(logits, dim=-1)
            naechstes = torch.multinomial(wahrscheinlichkeiten, num_samples=1)
        idx = torch.cat([idx, naechstes], dim=1)
        if naechstes.item() == modell.konfig.eos_id:
            grund = "Endtoken"
            break

    if grund == "Kontextgrenze" and not return_info:
        warnings.warn("Generierung an der Kontextgrenze beendet; nichts wurde abgeschnitten.")
    ergebnis = idx[0].tolist()
    return (ergebnis, grund) if return_info else ergebnis


def antworte(frage, temperatur=0.7, max_neu=64, top_k=50, return_info=False):
    prompt = f"<|im_start|>user\n{frage}<|im_end|>\n<|im_start|>assistant\n"
    ids = tokenizer.encode(prompt).ids
    alle, grund = erzeuge(modell, ids, max_neu, temperatur, top_k, return_info=True)
    text = tokenizer.decode(alle[len(ids):], skip_special_tokens=True).strip()
    if grund == "Kontextgrenze" and not return_info:
        warnings.warn("Antwort an der Kontextgrenze beendet.")
    return (text, grund) if return_info else text


### Beobachtung: Was die Temperatur macht

Vergleichen Sie dieselbe englische Frage bei mehreren Temperaturen. Temperatur 0
wählt deterministisch das wahrscheinlichste zulässige Token (Greedy-Decoding).
Positive Temperaturen verändern die Verteilung vor dem Sampling. Höhere Werte
flachen sie ab; welche Wirkung das auf Antwortlänge und Endtoken hat, hängt von
den konkreten Logits ab. Der ausgegebene Abbruchgrund unterscheidet ein gelerntes
Ende vom Tokenlimit und von der Kontextgrenze.

In [ ]:
frage = "hello guppy"
for temperatur in (0.0, 0.1, 0.7, 1.5):
    print(f"T = {temperatur}:")
    for _ in range(2):
        text, grund = antworte(frage, temperatur=temperatur, return_info=True)
        print(f"    {text} [{grund}]")

## 7. Gespräch mit dem Modell

Jede Frage steht für sich — das Modell hat kein Gedächtnis über Nachrichten
hinweg, es sieht immer nur den aktuellen Prompt. Mehrturn-Dialoge wären eine
eigene Übung: Man müsste die vorherigen Runden mit in den Prompt schreiben und
darauf achten, dass die 128 Tokens nicht überlaufen.

In [ ]:
fragen = [
    "hello guppy", "are you hungry", "it's very hot today",
    "how is the water", "do you like bubbles", "what is the internet",
    "do you sometimes feel lonely", "the cat is watching you",
    "tell me a joke", "what do you dream about",
    "what is the meaning of life", "goodnight guppy",
]

for frage in fragen:
    text, grund = antworte(frage, return_info=True)
    print(f"{frage:<30} -> {text} [{grund}]")


## 8. Speichern und Laden

Der Checkpoint enthält Gewichte, Architekturkonfiguration und den zugehörigen
Tokenizer samt SHA-256-Prüfsumme. Ein erneutes Tokenizertraining überschreibt
zwar `data/tokenizer.json`, verändert aber nicht den Tokenizer im gespeicherten
Checkpoint. Beim Laden verwenden wir immer das darin gebundene Paar.

Alte Checkpoints ohne Tokenizer werden mit verständlicher Meldung abgewiesen;
nach der Alphabetkorrektur ist ohnehin ein neuer Trainingslauf nötig.
Der Checkpoint dient der Inferenz, nicht dem exakten Fortsetzen einer
unterbrochenen Optimierung; dafür wären unter anderem Optimiererzustand und RNG-Zustände nötig.

Diese Fassung speichert unter `bestes_modell_b.pt`, damit der Checkpoint aus
`guppylm_lehrnotebook.ipynb` erhalten bleibt. Das Format ist identisch: Für den
Export nach LM Studio genügt es, im Export-Notebook den Dateinamen anzupassen.

In [ ]:
modell, tokenizer, ckpt = lade_checkpoint(WORK / "checkpoints" / "bestes_modell_b.pt", DEVICE)
print(f"Geladen aus Schritt {ckpt['schritt']}: {zaehle(modell):,} Parameter")
print(f"Checkpoint: {WORK / 'checkpoints' / 'bestes_modell_b.pt'}")
text, grund = antworte("hello guppy", temperatur=0, return_info=True)
print(f"\nProbe: {text} [{grund}]")

---

### Weiterarbeiten

- **Architektur.** Vergleichen Sie getrennt `n_layers = 2` und `n_heads = 1`
  mit dem Ausgangsmodell. Initialisieren und trainieren Sie jeweils neu. Bei
  gleicher Modellbreite verändert die Kopfzahl nicht die Zahl der linearen
  Attention-Parameter; weniger Blöcke reduzieren dagegen Parameter und Rechentiefe.
- **Tokenizer.** Setzen Sie die Zielgröße in `GRUNDKONFIG` auf 512. Starten Sie
  danach ab dem Tokenizertraining neu. Vergleichen Sie die Sequenzlängen und
  prüfen Sie die Kontextgrenze. Bewahren Sie Vergleichscheckpoints getrennt auf.
- **Padding.** Begründen Sie, warum rechtes Padding hier keine zusätzliche
  Attention-Maske erfordert. Was würde sich bei linkem Padding ändern?
- **Positionen.** Vergleichen Sie optional ein neu trainiertes Modell ohne
  Positions-Embeddings. Formulieren Sie zunächst eine Hypothese; ein bestimmtes
  Fehlerbild ist auf diesem einfachen Datensatz nicht garantiert.

- **Vergleich mit Fassung A.** Leitfragen zum Arbeitsauftrag:
  - An welcher Stelle unterscheidet sich der Code? Nutzen Sie einen Dateivergleich.
  - Warum zählt die Parametertabelle die Positionswerte weiterhin mit, obwohl
    sie sich im Training nicht ändern? Was bedeutet `requires_grad`?
  - Vergleichen Sie Verlustkurven, besten Eval-Verlust und Antworten beider Fassungen.
    Ist ein Unterschied bei einem Lauf je Fassung belastbar?
  - Vergleichen Sie die Heatmaps nach dem Training. Welche Positionen hat
    Fassung A überhaupt trainiert? Denken Sie an das 100-%-Quantil aus Abschnitt 2.
  - Warum lässt sich auch diese Fassung ohne Änderung am Export als GPT-2 ausdrücken?
- **Skalierung.** Trainieren Sie mit `POS_ALPHA = 1.0` neu und vergleichen Sie.
  Formulieren Sie vorher eine Hypothese anhand der Begründung in 4.4.

Die weiterführenden Experimente sind in `guppylm_aufgaben.md` beschrieben.